# Visual reasoning: новый fit и final_submission.csv

Кандидат B: frozen ImageNet ResNet-50 V2 layer3 → pooled cosine head + обучаемый с нуля bidirectional GRU (embedding 200, hidden 256). BCE, без аугментаций. Подписи канонизируются в формы train (`data.canonicalize`), оценки нормализуются внутри пар близнецов (`twin_normalize`).

На официальном holdout: mean AUC **0.605125**, sample STD **0.002294** по seeds 42/43/44 (MPS; не cross-validation). FiLM не выиграл сравнение. Финал: seed 42, train+val, **19 эпох** — округлённая вверх медиана best epoch [20, 19, 19]. Это новый fit, не загрузка contest checkpoint.

Запуск: **Restart & Run All**, интернет выключен. Приложить исходники `src/`, `configs/`, supplied data и разрешённые ImageNet-веса. Автозагрузок и установки пакетов здесь нет. Способ офлайн-доставки весов при лимите пакета требует подтверждения организаторов. Kaggle Default GPU cold proof пока не выполнен; локальный MPS-запуск не заменяет его.


In [ ]:
import os
import sys
import time
import shutil
from pathlib import Path

notebook_start = time.monotonic()
# Локально — корень проекта или notebooks/. На Kaggle — один приложенный проект.
project_source = Path(".")
if not Path("src/train.py").is_file():
    if Path("../src/train.py").is_file():
        os.chdir("..")
    else:
        projects = [p.parents[1] for p in Path("../input").glob("**/src/train.py")
                    if (p.parents[1] / "configs/config.yaml").is_file()]
        if len(projects) != 1:
            raise RuntimeError("Нужен ровно один приложенный проект с src/ и configs/ либо запуск из его корня")
        project_source = projects[0]
        for folder in ("src", "configs"):
            shutil.copytree(project_source / folder, folder, ignore=shutil.ignore_patterns("__pycache__"))
sys.path.insert(0, str(Path.cwd()))


## Среда и seed
Нужны локально установленные torch, torchvision, Hydra ≥1.4.0rc3, NumPy, Polars, scikit-learn, Pillow и tqdm. `uv sync`/`pip install` не вызываются. Версии ниже — фактические версии текущего kernel, а не обещание совместимости с другим runtime.


In [ ]:
import random
import platform
import numpy as np
import torch
import torchvision
import hydra
import polars as pl
import sklearn
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

if tuple(int(p) for p in hydra.__version__.split(".")[:2]) < (1, 4):
    raise RuntimeError("Нужна офлайн-установка Hydra >=1.4.0rc3: src.train использует execution whitelist")
with initialize_config_dir(config_dir=str(Path("configs").resolve())):
    cfg = compose(config_name="config", overrides=["experiment=final"])
assert cfg.data.train_on_val and cfg.train.predict_test and cfg.train.early_stopping is None
assert cfg.model.init_from is None and cfg.data.canonicalize and cfg.train.postprocess.twin_normalize
os.environ["PYTHONHASHSEED"] = str(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
torch.use_deterministic_algorithms(True, warn_only=True)
versions = {"python": platform.python_version(), "torch": torch.__version__,
            "torchvision": torchvision.__version__, "hydra": hydra.__version__,
            "numpy": np.__version__, "polars": pl.__version__, "scikit-learn": sklearn.__version__}
accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ("Apple MPS" if torch.backends.mps.is_available() else "CPU")
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") and not torch.cuda.is_available():
    raise RuntimeError("На Kaggle требуется включить GPU; CPU-запуск не является финальной проверкой")
print(versions, "seed", cfg.seed, "hardware", accelerator)
if not torch.cuda.is_available():
    print("Локальная проверка механики: результат не подтверждает лимит Kaggle Default GPU")


## Только явно приложенные ресурсы
Ожидаются `data/{train,val,test}.csv`, `data/sample_submission.csv`, каталоги изображений `data/{train,val,test}` и `resources/torch/hub/checkpoints/<официальный ImageNet filename>`. Если данных/весов нет по этим путям, ищется единственный соответствующий ресурс среди приложенных `../input`; создаётся относительная ссылка. Test до fitting здесь не читается — проверяется только наличие файла.


In [ ]:
import hashlib
from urllib.parse import urlparse

csv_names = ("train.csv", "val.csv", "test.csv", "sample_submission.csv")
if not all((Path("data") / name).is_file() for name in csv_names):
    data_sources = [p.parent for p in Path("../input").glob("**/train.csv")
                    if all((p.parent / name).is_file() for name in csv_names)]
    if len(data_sources) != 1 or Path("data").exists():
        raise RuntimeError("Нужны supplied CSV в data/ или один однозначный приложенный каталог данных")
    Path("data").symlink_to(os.path.relpath(data_sources[0], Path(".")), target_is_directory=True)
for name in ("train", "val", "test"):
    if not (Path("data") / name).is_dir():
        raise FileNotFoundError(f"Нет supplied изображений: data/{name}")

weights = torchvision.models.get_model_weights(cfg.model.image_encoder.arch)[cfg.model.image_encoder.weights]
filename = Path(urlparse(weights.url).path).name
weight_path = Path("resources/torch/hub/checkpoints") / filename
if not weight_path.is_file():
    matches = list(Path("../input").glob(f"**/{filename}"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Приложите разрешённые ImageNet-веса: {weight_path}; скачивания не будет")
    weight_path.parent.mkdir(parents=True, exist_ok=True)
    weight_path.symlink_to(os.path.relpath(matches[0], weight_path.parent))
with weight_path.open("rb") as f:
    digest = hashlib.file_digest(f, "sha256").hexdigest()
if not digest.startswith(filename.rsplit("-", 1)[1].split(".")[0]):
    raise ValueError("Хеш приложенных весов не совпадает с официальным torchvision filename")
os.environ["TORCH_HOME"] = "resources/torch"
source_files = [p for folder in ("src", "configs", "notebooks")
                for p in (project_source / folder).rglob("*")
                if p.is_file() and p.suffix in (".py", ".yaml", ".ipynb")]
if (project_source / "pyproject.toml").is_file():
    source_files.append(project_source / "pyproject.toml")
source_bytes = sum(p.stat().st_size for p in source_files)
if source_bytes > 20 * 1024**2:
    raise RuntimeError("Исходники превышают 20 MB")
print("ImageNet", filename, "sha256", digest, "source bytes", source_bytes)


## Обучение и inference
Один существующий CLI, без второго train-loop. Временный пустой feature cache гарантирует cold extraction; contest checkpoints/features не подключаются. Лимит fitting учитывает подготовку; незавершённый фиксированный refit считается ошибкой.


In [ ]:
import json
import subprocess
import tempfile

run_root = Path("artefacts/hydra") / cfg.name_exp
before = set(run_root.glob("*/results.json"))
Path("artefacts").mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(prefix="final_cold_", dir="artefacts") as feature_dir:
    relative_features = os.path.relpath(feature_dir, Path.cwd())
    command = [sys.executable, "-m", "src.train", "experiment=final", f"paths.features_dir={relative_features}"]
    print(" ".join(command))
    subprocess.run(command, check=True)
new_results = set(run_root.glob("*/results.json")) - before
if len(new_results) != 1:
    raise RuntimeError("Не найден единственный новый завершённый final refit")
result_path = new_results.pop()
result = json.loads(result_path.read_text())
if result["epochs_run"] != cfg.train.epochs or result["best_epoch"] != cfg.train.epochs or "val" in result:
    raise RuntimeError("Нужен полный фиксированный refit, не holdout score и не частичный запуск")


In [ ]:
import csv
import math

with Path("final_submission.csv").open(newline="") as f:
    reader = csv.DictReader(f)
    if reader.fieldnames != ["id", cfg.data.submission_target_col]:
        raise ValueError("Неверные колонки final_submission.csv")
    rows = list(reader)
with Path(cfg.data.sample_submission).open(newline="") as f:
    expected_ids = [row["id"] for row in csv.DictReader(f)]
ids = [row["id"] for row in rows]
probabilities = [float(row[cfg.data.submission_target_col]) for row in rows]
if ids != expected_ids or len(set(ids)) != len(ids):
    raise ValueError("Submission должен повторять все уникальные sample id в исходном порядке")
if not all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities):
    raise ValueError("Вероятности должны быть конечны и лежать в [0, 1]")
wall_min = (time.monotonic() - notebook_start) / 60
receipt = {"hardware": accelerator, "versions": versions, "seed": cfg.seed,
           "epochs": result["epochs_run"], "rows": len(rows), "wall_min": wall_min,
           "cache_min": result["cache_min"], "fit_min": result["fit_min"],
           "predict_min": result["predict_min"], "peak_cuda_memory_mb": result["peak_cuda_memory_mb"],
           "source_bytes": source_bytes, "results": str(result_path),
           "kaggle_environment": bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))}
Path("artefacts/notebook_run.json").write_text(json.dumps(receipt, indent=2))
print(receipt)
if wall_min > 50:
    raise RuntimeError("Полный cold запуск превысил 50 минут; кандидат не допущен этим запуском")
print("Создан final_submission.csv; допуск на Kaggle подтверждается только фактическим offline запуском на Default GPU")
